In [6]:
# Bài 3
import math

class TicTacToeMinimax:
    def __init__(self, size, win_len):
        self.size = size
        self.win_len = win_len
        self.board = [['.' for _ in range(size)] for _ in range(size)]
        self.bot = 'O'
        self.player = 'X'

    def display_board(self):
        header = "     " + "".join(f"{i:<3}" for i in range(self.size))
        print("\n" + header)

        print("    " + "---" * self.size + "--")

        for i, row in enumerate(self.board):
            row_str = "".join(f"{cell:<3}" for cell in row)
            print(f"{i:2} | {row_str}|")

        print("    " + "---" * self.size + "--")

    def evaluate_line(self, r, c, dr, dc):
        b_count, p_count = 0, 0
        for i in range(self.win_len):
            nr, nc = r + dr*i, c + dc*i
            if 0 <= nr < self.size and 0 <= nc < self.size:
                if self.board[nr][nc] == self.bot: b_count += 1
                elif self.board[nr][nc] == self.player: p_count += 1
            else: return 0
        if b_count > 0 and p_count > 0: return 0
        if b_count > 0: return 10**b_count
        if p_count > 0: return -(10**(p_count + 1))
        return 0

    def heuristic(self):
        score = 0
        for r in range(self.size):
            for c in range(self.size):
                for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                    score += self.evaluate_line(r, c, dr, dc)
        return score

    def get_candidate_moves(self):
        candidates = set()
        has_piece = False
        for r in range(self.size):
            for c in range(self.size):
                if self.board[r][c] != '.':
                    has_piece = True
                    for dr in range(-1, 2):
                        for dc in range(-1, 2):
                            nr, nc = r + dr, c + dc
                            if 0 <= nr < self.size and 0 <= nc < self.size and self.board[nr][nc] == '.':
                                candidates.add((nr, nc))
        if not has_piece: return [(self.size // 2, self.size // 2)]
        return list(candidates)

    def minimax(self, depth, is_maximizing):
        h = self.heuristic()
        if abs(h) >= 10**self.win_len or depth == 0:
            return h

        moves = self.get_candidate_moves()
        if is_maximizing:
            best_score = -math.inf
            for r, c in moves:
                self.board[r][c] = self.bot
                score = self.minimax(depth - 1, False)
                self.board[r][c] = '.'
                best_score = max(score, best_score)
            return best_score
        else:
            best_score = math.inf
            for r, c in moves:
                self.board[r][c] = self.player
                score = self.minimax(depth - 1, True)
                self.board[r][c] = '.'
                best_score = min(score, best_score)
            return best_score

    def play(self):
        print(f"--- BÀI 3: MINIMAX {self.size}x{self.size} ---")
        while True:
            self.display_board()
            try:
                move = input("Lượt bạn (X) - Nhập 'hàng cột': ").split()
                r, c = int(move[0]), int(move[1])
                if self.board[r][c] != '.': continue
                self.board[r][c] = self.player
            except: continue

            if abs(self.heuristic()) >= 10**self.win_len:
                self.display_board(); print("BẠN THẮNG!"); break

            best_score, best_move = -math.inf, None
            for r, c in self.get_candidate_moves():
                self.board[r][c] = self.bot
                score = self.minimax(1, False) #depth = 1
                self.board[r][c] = '.'
                if score > best_score:
                    best_score, best_move = score, (r, c)

            if best_move: self.board[best_move[0]][best_move[1]] = self.bot
            if self.heuristic() >= 10**self.win_len:
                self.display_board(); print("BOT THẮNG!"); break

# Run
try:
    n = int(input("Nhập kích thước N: "))
    win = 5 if n >= 5 else 3
    game = TicTacToeMinimax(size=n, win_len=win)
    game.play()
except:
    print("Có lỗi xảy ra!")

Nhập kích thước N: 8
--- BÀI 3: MINIMAX 8x8 ---

     0  1  2  3  4  5  6  7  
    --------------------------
 0 | .  .  .  .  .  .  .  .  |
 1 | .  .  .  .  .  .  .  .  |
 2 | .  .  .  .  .  .  .  .  |
 3 | .  .  .  .  .  .  .  .  |
 4 | .  .  .  .  .  .  .  .  |
 5 | .  .  .  .  .  .  .  .  |
 6 | .  .  .  .  .  .  .  .  |
 7 | .  .  .  .  .  .  .  .  |
    --------------------------
Lượt bạn (X) - Nhập 'hàng cột': 4 4

     0  1  2  3  4  5  6  7  
    --------------------------
 0 | .  .  .  .  .  .  .  .  |
 1 | .  .  .  .  .  .  .  .  |
 2 | .  .  .  .  .  .  .  .  |
 3 | .  .  .  .  O  .  .  .  |
 4 | .  .  .  .  X  .  .  .  |
 5 | .  .  .  .  .  .  .  .  |
 6 | .  .  .  .  .  .  .  .  |
 7 | .  .  .  .  .  .  .  .  |
    --------------------------
Lượt bạn (X) - Nhập 'hàng cột': 3 3

     0  1  2  3  4  5  6  7  
    --------------------------
 0 | .  .  .  .  .  .  .  .  |
 1 | .  .  .  .  .  .  .  .  |
 2 | .  .  .  .  .  .  .  .  |
 3 | .  .  .  X  O  .  .  .  |
 4 | .  .  .

In [5]:
# Bài 4
import math

class TicTacToeAlphaBeta:
    def __init__(self, size, win_len):
        self.size = size
        self.win_len = win_len
        self.board = [['.' for _ in range(size)] for _ in range(size)]
        self.bot = 'O'
        self.player = 'X'

    def display_board(self):

        header = "     " + "".join(f"{i:<3}" for i in range(self.size))
        print("\n" + header)
        print("    " + "---" * self.size + "--")
        for i, row in enumerate(self.board):
            row_str = "".join(f"{cell:<3}" for cell in row)
            print(f"{i:2} | {row_str}|")
        print("    " + "---" * self.size + "--")

    def evaluate_line(self, r, c, dr, dc):
        b_count, p_count = 0, 0
        for i in range(self.win_len):
            nr, nc = r + dr*i, c + dc*i
            if 0 <= nr < self.size and 0 <= nc < self.size:
                if self.board[nr][nc] == self.bot: b_count += 1
                elif self.board[nr][nc] == self.player: p_count += 1
            else: return 0
        if b_count > 0 and p_count > 0: return 0

        if b_count > 0: return 10**b_count

        if p_count > 0: return -(10**(p_count + 1))
        return 0

    def heuristic(self):
        score = 0
        for r in range(self.size):
            for c in range(self.size):
                for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                    score += self.evaluate_line(r, c, dr, dc)
        return score

    def get_candidate_moves(self):
        candidates = set()
        has_piece = False
        for r in range(self.size):
            for c in range(self.size):
                if self.board[r][c] != '.':
                    has_piece = True
                    for dr in range(-1, 2):
                        for dc in range(-1, 2):
                            nr, nc = r + dr, c + dc
                            if 0 <= nr < self.size and 0 <= nc < self.size and self.board[nr][nc] == '.':
                                candidates.add((nr, nc))
        if not has_piece: return [(self.size // 2, self.size // 2)]
        return list(candidates)

    def alpha_beta(self, depth, alpha, beta, is_maximizing):
        h = self.heuristic()
        # Dừng
        if abs(h) >= 10**self.win_len or depth == 0:
            return h

        moves = self.get_candidate_moves()
        if is_maximizing:
            max_val = -math.inf
            for r, c in moves:
                self.board[r][c] = self.bot
                val = self.alpha_beta(depth - 1, alpha, beta, False)
                self.board[r][c] = '.'
                max_val = max(max_val, val)
                alpha = max(alpha, val)
                if beta <= alpha:
                    break
            return max_val
        else:
            min_val = math.inf
            for r, c in moves:
                self.board[r][c] = self.player
                val = self.alpha_beta(depth - 1, alpha, beta, True)
                self.board[r][c] = '.'
                min_val = min(min_val, val)
                beta = min(beta, val)
                if beta <= alpha:
                    break
            return min_val

    def play(self):
        print(f"--- BÀI 4: ALPHA-BETA  {self.size}x{self.size} ---")
        while True:
            self.display_board()
            try:
                move = input("Lượt bạn (X) - Nhập 'hàng cột': ").split()
                r, c = int(move[0]), int(move[1])
                if self.board[r][c] != '.': continue
                self.board[r][c] = self.player
            except: continue

            if abs(self.heuristic()) >= 10**self.win_len:
                self.display_board(); print("BẠN THẮNG!"); break

            print("Bot đang suy nghĩ...")
            best_score, best_move = -math.inf, None
            # depth = 2
            for r, c in self.get_candidate_moves():
                self.board[r][c] = self.bot
                score = self.alpha_beta(2, -math.inf, math.inf, False)
                self.board[r][c] = '.'
                if score > best_score:
                    best_score, best_move = score, (r, c)

            if best_move: self.board[best_move[0]][best_move[1]] = self.bot
            if self.heuristic() >= 10**self.win_len:
                self.display_board(); print("BOT THẮNG!"); break

# INPUT and RUN
try:
    n = int(input("Nhập kích thước N: "))
    win = 5 if n >= 5 else 3
    game = TicTacToeAlphaBeta(size=n, win_len=win)
    game.play()
except:
    print("Lỗi nhập liệu!")

Nhập kích thước N: 10
--- BÀI 4: ALPHA-BETA  10x10 ---

     0  1  2  3  4  5  6  7  8  9  
    --------------------------------
 0 | .  .  .  .  .  .  .  .  .  .  |
 1 | .  .  .  .  .  .  .  .  .  .  |
 2 | .  .  .  .  .  .  .  .  .  .  |
 3 | .  .  .  .  .  .  .  .  .  .  |
 4 | .  .  .  .  .  .  .  .  .  .  |
 5 | .  .  .  .  .  .  .  .  .  .  |
 6 | .  .  .  .  .  .  .  .  .  .  |
 7 | .  .  .  .  .  .  .  .  .  .  |
 8 | .  .  .  .  .  .  .  .  .  .  |
 9 | .  .  .  .  .  .  .  .  .  .  |
    --------------------------------
Lượt bạn (X) - Nhập 'hàng cột': 4 5
Bot đang suy nghĩ...

     0  1  2  3  4  5  6  7  8  9  
    --------------------------------
 0 | .  .  .  .  .  .  .  .  .  .  |
 1 | .  .  .  .  .  .  .  .  .  .  |
 2 | .  .  .  .  .  .  .  .  .  .  |
 3 | .  .  .  .  .  .  .  .  .  .  |
 4 | .  .  .  .  .  X  .  .  .  .  |
 5 | .  .  .  .  O  .  .  .  .  .  |
 6 | .  .  .  .  .  .  .  .  .  .  |
 7 | .  .  .  .  .  .  .  .  .  .  |
 8 | .  .  .  .  .  .  .  .  .  .  |


In [3]:
import math
import time

class OAnQuan_Pro:
    def __init__(self):
        # set up
        self.board = [5, 5, 5, 5, 5, 10, 5, 5, 5, 5, 5, 10]
        self.score = [0, 0]
        self.quan_indices = [5, 11]

    def hien_thi(self):
        b = self.board
        print("\n" + "="*45)
        print(f" ĐIỂM |  NGƯỜI: {self.score[0]:2}  |  MÁY: {self.score[1]:2}")
        print("="*45)
        print(f"        ({b[10]:>2}) ({b[9]:>2}) ({b[8]:>2}) ({b[7]:>2}) ({b[6]:>2})")
        print(f"  [{b[11]:>2}] {' '*25} [{b[5]:>2}]")
        print(f"        ({b[0]:>2}) ({b[1]:>2}) ({b[2]:>2}) ({b[3]:>2}) ({b[4]:>2})")
        print("="*45)
        print(" CHỈ SỐ:  0     1     2     3     4    (Nhập 'end' để dừng)")

    def rai_quan(self, board, score, pos, direction, player):
        temp_board = list(board)
        temp_score = list(score)
        hand = temp_board[pos]
        temp_board[pos] = 0
        curr = pos

        while True:
            while hand > 0:
                curr = (curr + direction) % 12
                temp_board[curr] += 1
                hand -= 1

            next_p = (curr + direction) % 12
            if temp_board[next_p] > 0 and next_p not in self.quan_indices:
                hand = temp_board[next_p]
                temp_board[next_p] = 0
                curr = next_p
                continue
            elif temp_board[next_p] == 0:
                target = (next_p + direction) % 12
                if temp_board[target] > 0:
                    while temp_board[target] > 0 and temp_board[next_p] == 0:
                        temp_score[player] += temp_board[target]
                        temp_board[target] = 0
                        next_p = (target + direction) % 12
                        target = (next_p + direction) % 12
                break
            else: break
        return temp_board, temp_score

    def get_valid_moves(self, board, player):
        start, end = (0, 5) if player == 0 else (6, 11)
        return [(i, d) for i in range(start, end) for d in [1, -1] if board[i] > 0]

    def alpha_beta(self, board, score, depth, alpha, beta, is_max):
        if depth == 0 or (board[5] == 0 and board[11] == 0):
            return score[1] - score[0]

        valid_moves = self.get_valid_moves(board, 1 if is_max else 0)
        if not valid_moves: return score[1] - score[0]

        if is_max:
            v = -math.inf
            for m in valid_moves:
                nb, ns = self.rai_quan(board, score, m[0], m[1], 1)
                v = max(v, self.alpha_beta(nb, ns, depth-1, alpha, beta, False))
                alpha = max(alpha, v)
                if beta <= alpha: break
            return v
        else:
            v = math.inf
            for m in valid_moves:
                nb, ns = self.rai_quan(board, score, m[0], m[1], 0)
                v = min(v, self.alpha_beta(nb, ns, depth-1, alpha, beta, True))
                beta = min(beta, v)
                if beta <= alpha: break
            return v

    def play(self):
        print("---BÀI 5: TRÒ CHƠI Ô ĂN QUAN  ---")
        while self.board[5] > 0 or self.board[11] > 0:
            self.hien_thi()

            # LƯỢT NGƯỜI
            cmd = input("\nLượt bạn - Chọn ô (0-4) hoặc 'end': ").lower().strip()
            if cmd == 'end':
                print("\n>>> BẠN ĐÃ CHỌN KẾT THÚC TRÒ CHƠI SỚM.")
                break

            try:
                choice = int(cmd)
                huong = int(input("Hướng (1: Phải, -1: Trái): "))
                if (choice, huong) in self.get_valid_moves(self.board, 0):
                    self.board, self.score = self.rai_quan(self.board, self.score, choice, huong, 0)
                else:
                    print("!!! Nước đi không hợp lệ, vui lòng chọn lại.")
                    continue
            except:
                print("!!! Vui lòng nhập số hoặc 'end'.")
                continue

            # LƯỢT MÁY
            print("Máy đang suy nghĩ...")
            best_v, best_m = -math.inf, None
            for m in self.get_valid_moves(self.board, 1):
                nb, ns = self.rai_quan(self.board, self.score, m[0], m[1], 1)
                val = self.alpha_beta(nb, ns, 3, -math.inf, math.inf, False)
                if val > best_v:
                    best_v, best_m = val, m

            if best_m:
                self.board, self.score = self.rai_quan(self.board, self.score, best_m[0], best_m[1], 1)
                print(f"-> Máy đã đi ô {best_m[0]} hướng {best_m[1]}")
            else: break

        print("\n" + "*"*20)
        print("KẾT QUẢ CUỐI CÙNG:")
        self.hien_thi()
        if self.score[0] > self.score[1]: print("CHÚC MỪNG! BẠN THẮNG!")
        elif self.score[0] < self.score[1]: print("MÁY THẮNG! HẸN GẶP LẠI LẦN SAU.")
        else: print("KẾT QUẢ HÒA!")

if __name__ == "__main__":
    game = OAnQuan_Pro()
    game.play()

---BÀI 5: TRÒ CHƠI Ô ĂN QUAN  ---

 ĐIỂM |  NGƯỜI:  0  |  MÁY:  0
        ( 5) ( 5) ( 5) ( 5) ( 5)
  [10]                           [10]
        ( 5) ( 5) ( 5) ( 5) ( 5)
 CHỈ SỐ:  0     1     2     3     4    (Nhập 'end' để dừng)

Lượt bạn - Chọn ô (0-4) hoặc 'end': 4
Hướng (1: Phải, -1: Trái): 1
Máy đang suy nghĩ...
-> Máy đã đi ô 8 hướng 1

 ĐIỂM |  NGƯỜI: 11  |  MÁY: 21
        ( 0) ( 9) ( 0) ( 8) ( 8)
  [ 0]                           [ 2]
        ( 0) ( 0) ( 8) ( 1) ( 2)
 CHỈ SỐ:  0     1     2     3     4    (Nhập 'end' để dừng)

Lượt bạn - Chọn ô (0-4) hoặc 'end': 2
Hướng (1: Phải, -1: Trái): -1
Máy đang suy nghĩ...
-> Máy đã đi ô 6 hướng 1

 ĐIỂM |  NGƯỜI: 11  |  MÁY: 35
        ( 4) ( 0) ( 0) ( 0) ( 0)
  [ 4]                           [ 5]
        ( 0) ( 4) ( 3) ( 4) ( 0)
 CHỈ SỐ:  0     1     2     3     4    (Nhập 'end' để dừng)

Lượt bạn - Chọn ô (0-4) hoặc 'end': 1
Hướng (1: Phải, -1: Trái): 1
Máy đang suy nghĩ...
-> Máy đã đi ô 10 hướng -1

 ĐIỂM |  NGƯỜI: 11  |  MÁY: 35
 

In [1]:
# Bài 5
import ipywidgets as widgets
from IPython.display import display, clear_output
import math

def check_win(board, p):
    size = len(board)
    for i in range(size):
        if all([board[i][j] == p for j in range(size)]) or \
           all([board[j][i] == p for j in range(size)]): return True
    if all([board[i][i] == p for i in range(size)]) or \
       all([board[i][size-i-1] == p for i in range(size)]): return True
    return False

def is_full(board):
    return all([cell != '.' for row in board for cell in row])

def minimax(board, depth, alpha, beta, is_maximizing):
    if check_win(board, 'O'): return 1
    if check_win(board, 'X'): return -1
    if is_full(board): return 0
    if is_maximizing:
        v = -math.inf
        for r in range(len(board)):
            for c in range(len(board)):
                if board[r][c] == '.':
                    board[r][c] = 'O'; res = minimax(board, depth+1, alpha, beta, False); board[r][c] = '.'
                    v = max(v, res); alpha = max(alpha, v)
                    if beta <= alpha: break
        return v
    else:
        v = math.inf
        for r in range(len(board)):
            for c in range(len(board)):
                if board[r][c] == '.':
                    board[r][c] = 'X'; res = minimax(board, depth+1, alpha, beta, True); board[r][c] = '.'
                    v = min(v, res); beta = min(beta, v)
                    if beta <= alpha: break
        return v

size = 3
board = [['.' for _ in range(size)] for _ in range(size)]

title = widgets.HTML("<h1>TIC TAC TOE AI</h1>")
status_label = widgets.Label(value="Lượt của bạn (X)", style={'font_weight': 'bold'})

grid = widgets.GridspecLayout(size, size, grid_gap='2px', layout=widgets.Layout(width='250px', margin='0 auto'))

def on_click(b):
    r, c = b.indices
    if board[r][c] == '.' and not check_win(board, 'X') and not check_win(board, 'O'):
        board[r][c] = 'X'
        b.description = 'X'
        b.style.button_color = '#4CAF50'
        b.style.font_weight = 'bold'

        if check_win(board, 'X'):
            status_label.value = "KẾT QUẢ: BẠN THẮNG! 🎉"
            return
        if is_full(board):
            status_label.value = "KẾT QUẢ: HÒA! 🤝"
            return

        status_label.value = "Máy đang suy nghĩ..."
        move = None
        best_s = -math.inf
        for row in range(size):
            for col in range(size):
                if board[row][col] == '.':
                    board[row][col] = 'O'
                    score = minimax(board, 0, -math.inf, math.inf, False)
                    board[row][col] = '.'
                    if score > best_s: best_s = score; move = (row, col)

        if move:
            board[move[0]][move[1]] = 'O'
            target_btn = grid[move[0], move[1]]
            target_btn.description = 'O'
            target_btn.style.button_color = '#FF5722' # Cam đỏ
            target_btn.style.font_weight = 'bold'

            if check_win(board, 'O'):
                status_label.value = "KẾT QUẢ: MÁY THẮNG! 💀"
            else:
                status_label.value = "Lượt của bạn (X)"

for r in range(size):
    for c in range(size):
        btn = widgets.Button(description='', layout=widgets.Layout(height='80px', width='80px'))
        btn.indices = (r, c)
        btn.on_click(on_click)
        grid[r, c] = btn

ui = widgets.VBox([title, status_label, grid], layout=widgets.Layout(align_items='center', background_color='#f0f0f0', padding='20px'))

display(ui)